In [1]:
# Parameters
BATCH_MODE = "true"


# SC-14-Formal-Vérification - Vérification Formelle

[<< Précédent : Fuzz & Invariants](SC-13-Fuzz-Invariants-Python.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Zero-Knowledge Proofs >>](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs-Python.ipynb)

***

## Objectifs d'apprentissage

1. Comprendre les principes de la **vérification formelle**
2. Utiliser **Certora Prover** et le langage CVL
3. Ecrire des **spécifications** et des **règles**
4. Vérifier des invariants mathématiques

### Prérequis

- [SC-12](SC-12-Foundry-Testing-Python.ipynb) et [SC-13](SC-13-Fuzz-Invariants-Python.ipynb) complétés
- Notions de logique formelle (optionnel)
- Foundry installé, accès à Certora ou Halmos recommandé

### Durée estimée : 50 minutes

***

## 1. Introduction à la Vérification Formelle

La vérification formelle prouve mathématiquement la correction d'un programme.

In [2]:
# Concepts de verification formelle
print("""
VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)
""")


VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)



### Interpretation : Vérification formelle vs Tests

**Résultat obtenu** : Comparaison des approches de validation.

| Critère | Tests (unitaires/fuzz) | Vérification formelle |
|---------|------------------------|----------------------|
| **Couverture** | Partielle (échantillonnage) | Complète (exhaustive) |
| **Méthodologie** | Exemples concrets | Preuves mathématiques |
| **Complexité** | Simple à mettre en place | Requires expertise |
| **Coût** | Faible | Élevé (temps, ressources) |
| **Faux positifs** | Non | Possibles (specs incorrectes) |
| **Faux négatifs** | Possibles (cas non testés) | Non (si spec correcte) |

**Points clés** :
- La vérification formelle prouve qu'une propriété est vraie pour TOUTES les entrées possibles
- Les tests ne peuvent que vérifier des cas spécifiques
- Certora Prover est un outil commercial puissant utilisant le langage CVL
- SMTChecker est une alternative gratuite intégrée au compilateur Solidity
- Les deux approches sont complémentaires : tests pour développement rapide, vérification formelle pour contrats critiques

***

## 2. Certora Prover

Certora utilise le CVL (Certora Vérification Language).

In [3]:
# Contrat Solidity a verifier
SOL_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}
'''

print("Contrat Vault a verifier:")
print(SOL_CONTRACT)

Contrat Vault a verifier:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}



Spécification formelle CVL (Certora Vérification Language) définissant les propriétés à vérifier sur le contrat Vault.

In [4]:
# Specification CVL pour Vault
CVL_SPEC = '''
// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);
    
    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();
    
    // Verifier le resultat
    assert balances(user) == balanceBefore + amount;
}

// Regle: withdraw reduit correctement la balance
rule withdrawDecreasesBalance(address user, uint256 amount) {
    require amount > 0;
    require balances(user) >= amount;
    
    uint256 balanceBefore = balances(user);
    uint256 totalBefore = totalDeposits();
    
    env.msgSender = user;
    withdraw(amount);
    
    assert balances(user) == balanceBefore - amount;
    assert totalDeposits() == totalBefore - amount;
}
'''

print("Specification CVL:")
print(CVL_SPEC)

Specification CVL:

// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);

    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();

    // Verifier le resultat
  

### Interpretation : Spécification CVL - Invariants et règles

**Résultat obtenu** : Langage de spécification formelle pour Certora Prover.

| Concept | Syntaxe CVL | Signification |
|---------|-------------|---------------|
| **Invariant** | `invariant nom() condition` | Propriété toujours vraie |
| **Règle** | `rule nom(params) { require...; env...; call(); assert...; }` | Comportement attendu |
| **Sum** | `sum(x => f(x))` | Quantification universelle |

**Points clés** :
- `totalDepositsEqSumOfBalances()` : invariant de conservation (somme des balances = total)
- `balancesNonNegative()` : invariant de sécurité (pas de balances négatives)
- `depositIncreasesBalance()` : règle de comportement (après deposit, balance augmente)
- `env.msgSender` et `env.msgValue` permettent de simuler l'environnement d'appel

***

## 3. Commandes Certora

Exécution du vérifier.

In [5]:
# Commandes Certora
print("""
INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance
""")


INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance



***
## 3b. SMTChecker - Vérification native Solidity

Contrairement à Certora (outil externe, commercial), **SMTChecker** est intégré au compilateur Solidity. Il utilise des solveurs SMT (Z3, CVC5) pour vérifier automatiquement des invariants.

| Aspect | Certora Prover | SMTChecker |
|--------|---------------|------------|
| Installation | `npm install -g @certora/certora-cli` | Inclus dans `solc` |
| Langage spec | CVL séparé | Assertions Solidity natives |
| Coût | Commercial (API key) | Gratuit |
| Couverture | Très complet | Limité (pas de boucles infinies) |
| Model checking | Interactif | Automatique |

Activation : `pragma experimental SMTChecker;` dans le contrat.

In [6]:
# Contrat Vault avec SMTChecker - detection d'un bug d'overflow
vault_smt_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract VaultSMT {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    /// @custom:invariant totalDeposits == sum(balances)
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;

        // Assertion SMTChecker : le solde ne peut pas etre negatif
        assert(balances[msg.sender] >= 0);

        payable(msg.sender).transfer(amount);
    }

    // Assertion : totalDeposits toujours coherent
    function checkInvariant() public view {
        assert(totalDeposits >= 0);
    }
}
"""

print("Contrat VaultSMT avec assertions SMTChecker :")
print(vault_smt_code)
print("\n---")
print("Points cles :")
print("1. 'pragma experimental SMTChecker' active le model checker")
print("2. Les 'assert()' sont verifiees pour TOUTES les executions possibles")
print("3. Si une assertion peut echouer, le compilateur genere un contre-exemple")

Contrat VaultSMT avec assertions SMTChecker :

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract VaultSMT {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    /// @custom:invariant totalDeposits == sum(balances)
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;

        // Assertion SMTChecker : le solde ne peut pas etre negatif
        assert(balances[msg.sender] >= 0);

        payable(msg.sender).transfer(amount);
    }

    // Assertion : totalDeposits toujours coherent
    function checkInvariant() public view {
        assert(totalDeposits >= 0);
    }
}


---
Points cles :
1. 'pragma experimental SMTChecker' active le model ch

### Interpretation : SMTChecker - Model checking natif

**Résultat obtenu** : Activation du model checker Solidity avec détection potentielle de bugs.

| Élément | Rôle |
|---------|------|
| `pragma experimental SMTChecker` | Active le model checker natif |
| `assert(condition)` | Vérifie que la condition est TOUJOURS vraie |
| `@custom:Invariant` | Documentation des invariants attendus |

**Points clés** :
- Contrairement aux tests (qui échantillonnent), SMTChecker explore TOUTES les exécutions possibles
- Si une assertion peut échouer, le compilateur génère un contre-exemple concret
- Les limitations : pas de support complet pour les boucles, certains appels externes
- Avantage majeur : intégré au compilateur, gratuit, pas besoin de langage de spec externe

In [7]:
# Preparation du solveur SMT : quel binaire solc sait reellement verifier ?
#
# SMTChecker n'embarque pas de solveur : solc charge z3 a l'execution. Les binaires
# Windows n'exposent pas ce chemin de chargement, donc la verification n'y aboutit
# jamais, quelle que soit la DLL placee a cote du binaire. Le binaire Linux charge
# libz3.so.4.12, et rend alors de vrais verdicts.
#
# Cette cellule etablit le backend utilisable sur cette machine, et provisionne dans
# WSL le binaire solc et la bibliotheque z3 s'ils manquent (telechargement unique).
import os
import re
import shutil
import subprocess
import tempfile
import textwrap

SOLC_LINUX_URL = (
    "https://binaries.soliditylang.org/linux-amd64/solc-linux-amd64-v0.8.28+commit.7893614a"
)
Z3_URL = (
    "https://github.com/Z3Prover/z3/releases/download/z3-4.12.6/z3-4.12.6-x64-glibc-2.35.zip"
)
SOLC_WSL = "solc-0.8.28"
Z3_LIB = "/tmp/z3lib/libz3.so.4.12"

CONTRAT_SONDE = textwrap.dedent(
    """\
    // SPDX-License-Identifier: MIT
    pragma solidity ^0.8.28;

    contract Sonde {
        function suivant(uint256 x) public pure returns (uint256) {
            return x + 1;
        }
    }
    """
)


def _wsl(script, timeout=900):
    """Execute un script bash dans WSL ; rend (code de sortie, sortie texte)."""
    res = subprocess.run(
        ["wsl", "-e", "bash", "-lc", script],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )
    return res.returncode, (res.stdout + res.stderr).strip()


def _solveur_a_repondu(sortie):
    """Vrai quand le diagnostic montre qu'un solveur SMT a reellement travaille."""
    return not (
        "not available" in sortie
        or "no Horn solver" in sortie
        or "no SMT solver" in sortie
    )


def _solc_natif():
    """Chemin du solc installe localement (solcx, puis PATH), ou None."""
    try:
        import solcx

        if hasattr(solcx, "get_executable"):
            return solcx.get_executable("0.8.28")
        dossier = os.path.join(solcx.get_solcx_install_folder(), "solc-v0.8.28")
        for nom in ("solc.exe", "solc"):
            candidat = os.path.join(dossier, nom)
            if os.path.isfile(candidat):
                return candidat
    except Exception:
        pass
    return shutil.which("solc")


def _sonde_natif(solc_bin):
    """Demande un verdict a un solc local ; None s'il ne sait pas verifier."""
    if not solc_bin:
        return None
    fichier = os.path.join(tempfile.mkdtemp(prefix="sc14_"), "sonde.sol")
    try:
        with open(fichier, "w", encoding="utf-8") as flux:
            flux.write(CONTRAT_SONDE)
        res = subprocess.run(
            [solc_bin, "--model-checker-engine", "all", fichier],
            capture_output=True,
            text=True,
        encoding="utf-8",
        errors="replace",
            timeout=300,
        )
        return (res.stderr + res.stdout).strip()
    except (OSError, subprocess.SubprocessError):
        return None
    finally:
        shutil.rmtree(os.path.dirname(fichier), ignore_errors=True)


def _provisionne_wsl():
    """Installe solc Linux et z3 4.12 dans WSL. Idempotent ; rend True si pret."""
    code, sortie = _wsl(
        f"test -x /tmp/{SOLC_WSL} && test -f {Z3_LIB} && echo PRET || echo MANQUANT"
    )
    if code != 0:
        return False
    if "PRET" in sortie:
        return True
    print("Provisionnement de WSL : binaire solc Linux + z3 4.12 (une seule fois)...")
    script = (
        "set -e; cd /tmp; "
        f"test -x {SOLC_WSL} || {{ curl -sS -L -o {SOLC_WSL} {SOLC_LINUX_URL}; chmod +x {SOLC_WSL}; }}; "
        f"if [ ! -f {Z3_LIB} ]; then "
        f"  curl -sS -L -o z3-4.12.6.zip {Z3_URL}; "
        "  unzip -oq z3-4.12.6.zip -d z3-smt; "
        f"  mkdir -p /tmp/z3lib; ln -sf $(find z3-smt -name libz3.so | head -1) {Z3_LIB}; "
        "fi; "
        f"test -f {Z3_LIB} && echo PROVISIONNE"
    )
    code, sortie = _wsl(script)
    derniere = sortie.splitlines()[-1] if sortie else "(aucune sortie)"
    print(f"[provisionnement : code {code}] {derniere}")
    return code == 0 and "PROVISIONNE" in sortie


def _sonde_wsl():
    """Demande un verdict au solc Linux dans WSL ; None si le solveur manque."""
    script = (
        f"export LD_LIBRARY_PATH=/tmp/z3lib; cd /tmp; "
        f"cat > sonde_sc14.sol <<'SOL14'\n{CONTRAT_SONDE}\nSOL14\n"
        f"./{SOLC_WSL} --model-checker-engine all sonde_sc14.sol 2>&1"
    )
    code, sortie = _wsl(script)
    return sortie if code == 0 else None


def choisir_backend():
    """Rend (type, description) du premier backend qui produit un verdict SMT."""
    solc_bin = _solc_natif()
    sortie = _sonde_natif(solc_bin)
    if sortie is not None and _solveur_a_repondu(sortie):
        return ("natif", f"{os.path.basename(solc_bin)} (solveur lie)")
    if shutil.which("wsl") and _provisionne_wsl():
        sortie = _sonde_wsl()
        if sortie is not None and _solveur_a_repondu(sortie):
            return ("wsl", f"{SOLC_WSL} + libz3.so.4.12 (WSL)")
    return (None, "aucun binaire solc capable de charger un solveur SMT")


BACKEND, DESCRIPTION_BACKEND = choisir_backend()
print(f"Backend de verification : {BACKEND or 'absent'} -- {DESCRIPTION_BACKEND}")
if BACKEND is None:
    print("La cellule suivante affichera alors le diagnostic brut de solc, qui nomme")
    print("lui-meme le solveur manquant : le carnet reste lisible et honnete.")


Backend de verification : wsl -- solc-0.8.28 + libz3.so.4.12 (WSL)


In [8]:
# Verification formelle : trois contrats, trois verdicts rendus par le compilateur
#
# L'arithmetique non verifiee et l'ordre des effets sont deux familles de defauts
# que SMTChecker sait atteindre. Le troisieme contrat montre le cas ou il ne conclut
# pas, et pourquoi : une condition peut rester non prouvee sans etre refutee.
vault_buggy_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;
contract VaultBuggy {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }
    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        payable(msg.sender).transfer(amount);  // transfert AVANT mise a jour de l'etat
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        assert(balances[msg.sender] >= 0);
    }
}
"""

compteur_non_verifie_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;
contract CompteurNonVerifie {
    uint256 public count;
    function bump() external {
        unchecked { count += 1; }
        assert(count > 0);
    }
}
"""


def executer_smtchecker(code_sol, etiquette):
    """Invoque solc --model-checker-engine all et rend le diagnostic brut."""
    print(f"=== {etiquette} ===")
    if BACKEND is None:
        print("[verification non executee : aucun solveur SMT disponible]\n")
        return None
    if BACKEND == "natif":
        chemin = "sc14_cible.sol"
        with open(chemin, "w", encoding="utf-8") as flux:
            flux.write(code_sol)
        try:
            res = subprocess.run(
                [_solc_natif(), "--model-checker-engine", "all", chemin],
                capture_output=True,
                text=True,
        encoding="utf-8",
        errors="replace",
                timeout=300,
            )
            sortie = (res.stderr + res.stdout).strip()
        finally:
            os.unlink(chemin)
    else:
        script = (
            f"export LD_LIBRARY_PATH=/tmp/z3lib; cd /tmp; "
            f"cat > cible_sc14.sol <<'SOL14'\n{code_sol}\nSOL14\n"
            f"./{SOLC_WSL} --model-checker-engine all cible_sc14.sol 2>&1"
        )
        _, sortie = _wsl(script)
    print(f"$ solc --model-checker-engine all cible_sc14.sol\n\n{sortie}\n")
    return sortie


def classer(sortie):
    """Traduit un diagnostic solc en verdict lisible."""
    if sortie is None:
        return "non executee"
    if not _solveur_a_repondu(sortie):
        return "solveur absent"
    constats = []
    prouvees = re.search(r"(\d+) verification condition\(s\) proved safe", sortie)
    if prouvees:
        constats.append(f"{prouvees.group(1)} condition(s) prouvee(s)")
    if "could not be proved" in sortie:
        constats.append("condition(s) non prouvee(s)")
    if "Counterexample" in sortie:
        constats.append("contre-exemple produit")
    return ", ".join(constats) or "aucune condition de verification"


verifications = [
    ("VaultSMT : effets avant interaction", vault_smt_code),
    ("CompteurNonVerifie : arithmetique unchecked", compteur_non_verifie_code),
    ("VaultBuggy : interaction avant effets", vault_buggy_code),
]
resultats = [
    (etiquette, classer(executer_smtchecker(code, etiquette)))
    for etiquette, code in verifications
]

print("=" * 78)
print(f"{'Contrat':<44} | Verdict SMTChecker")
print("-" * 78)
for etiquette, verdict in resultats:
    print(f"{etiquette:<44} | {verdict}")
print("=" * 78)


=== VaultSMT : effets avant interaction ===


$ solc --model-checker-engine all cible_sc14.sol

 --> cible_sc14.sol:4:1:
  |
4 | pragma experimental SMTChecker;
  | ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: CHC: 3 verification condition(s) proved safe! Enable the model checker option "show proved safe" to see all of them.

=== CompteurNonVerifie : arithmetique unchecked ===


$ solc --model-checker-engine all cible_sc14.sol

 --> cible_sc14.sol:4:1:
  |
4 | pragma experimental SMTChecker;
  | ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


 --> cible_sc14.sol:9:9:
  |
9 |         assert(count > 0);
  |         ^^^^^^^^^^^^^^^^^
Note: Counterexample:
  count = 0

Note: Callstack:
Note:

=== VaultBuggy : interaction avant effets ===


$ solc --model-checker-engine all cible_sc14.sol

 --> cible_sc14.sol:4:1:
  |
4 | pragma experimental SMTChecker;
  | ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


Info: CHC: 1 verification condition(s) proved safe! Enable the model checker option "show proved safe" to see all of them.

  --> cible_sc14.sol:14:9:
   |
14 |         payable(msg.sender).transfer(amount);  // transfert AVANT mise a jour de l'etat
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Note: Counterexample:
  amount = 0
  balances[msg.sender] = 32278
  totalDeposits = 0

Note: Callstack:
Note: 
Note that array aliasing is not supported, therefore all mapping information is erased after a mapping local variable/parameter is assigned.
You can re-introduce information using require().

Contrat                                      | Verdict SMTChecker
------------------------------------------------------------------------------
VaultSMT : effets avant interaction          | 3 condition(s) prouvee(s)
CompteurNonVerifie : arithm

### Interprétation : ce que le compilateur a réellement prouvé

Les trois contrats ne produisent pas le même verdict, et c'est tout l'intérêt du model checking : la réponse n'est pas « sûr / pas sûr », mais **prouvé**, **réfuté avec un contre-exemple**, ou **non conclu**.

| Contrat | Ce qui est demandé | Verdict rendu |
|---------|--------------------|---------------|
| `VaultSMT` | l'état est mis à jour avant l'interaction ; `balances` et `totalDeposits` ne peuvent pas devenir incohérents | **prouvé** : les conditions de vérification passent, aucun contre-exemple n'existe |
| `CompteurNonVerifie` | `assert(count > 0)` après une incrémentation `unchecked` | **réfuté** : le contre-exemple est `count = 0`, atteint quand l'incrémentation déborde et revient à zéro |
| `VaultBuggy` | `transfer` avant la mise à jour de l'état | **mixte** : une condition prouvée, une non prouvée, et un contre-exemple sur le transfert |

Trois lectures à retenir de ce tableau.

**Un contre-exemple est une exécution, pas un avertissement.** `CompteurNonVerifie` illustre le cas le plus net : `unchecked` désactive la vérification de débordement du compilateur, l'incrémentation d'un `uint256` à sa valeur maximale revient à zéro, et l'assertion tombe. Le contre-exemple `count = 0` est la preuve ; il n'y a rien à discuter.

**Le verdict de `VaultBuggy` porte sur autre chose que le défaut visé.** Le contre-exemple rendu est *Insufficient funds* sur `payable(msg.sender).transfer(amount)` : SMTChecker signale que le transfert peut échouer, pas que le contrat est réentrant. Le défaut d'ordre effets/interaction reste réel, mais **SMTChecker ne modélise pas la réentrance** — il raisonne sur un contrat isolé, pas sur les rappels qu'un appel externe peut déclencher. C'est une limite de l'outil, pas une absolution du contrat.

**« Non prouvé » n'est pas « faux ».** Les moteurs CHC et BMC ne rendent pas le même service :

| Moteur | Méthode | Ce qu'il rend |
|--------|---------|---------------|
| **CHC** (Horn clauses) | recherche d'invariants, raisonnement non borné | « prouvé sûr » ou « non prouvé » |
| **BMC** (bounded model checking) | déroulement borné des exécutions | un contre-exemple concret, ou rien dans la borne explorée |

Une condition « could not be proved » signifie que le solveur n'a pas trouvé d'invariant dans le temps imparti — ni preuve, ni réfutation. La traiter comme un feu vert est l'erreur classique.

**Pourquoi un solveur externe.** `solc` n'embarque pas z3 : il le charge à l'exécution (`libz3.so.4.12` sous Linux). Sans cette bibliothèque, le compilateur se contente d'avertir que l'analyse n'a pas eu lieu et sort en code 0 — un code de sortie nul ne dit donc **rien** sur la vérification.

| Aspect | SMTChecker | Certora Prover |
|--------|------------|----------------|
| **Intégration** | Natif dans solc | Outil externe (npm) |
| **Langage** | Solidity (`assert`) | CVL (spec séparée) |
| **Coût** | Gratuit | Commercial |
| **Couverture** | Limité (boucles, réentrance, appels externes) | Très complet |
| **Dépendance** | Solveur SMT lié au binaire solc | Solveur SMT interne |

**Limites à connaître** : pas de raisonnement sur les boucles non bornées, pas de réentrance, pas de composition inter-contrats. SMTChecker attrape donc en priorité les défauts **locaux** — débordements, divisions par zéro, assertions violées, invariants arithmétiques — et laisse à Certora, Foundry ou Echidna les propriétés globales.


### Exercice 2 : SMTChecker - Assertions pour un TokenVault

En vous basant sur l'exemple de `VaultSMT` ci-dessus, identifiez les assertions
SMTChecker manquantes dans un contrat `TokenVault` avec `transfer` et `withdraw`.

**Objectif** : Ajoutez les `assert()` manquants dans le code Solidity pour que
SMTChecker puisse vérifier les invariants de conservation et de non-négativité.

**Indice** :
- Dans `transfer()`, que doit-on vérifier sur `totalSupply` ?
- Dans `withdraw()`, que doit-on vérifier sur `balances[msg.sender]` ?
- Pensez au pattern "checks-effects-interactions"

**Étapes** :
1. Ecrire l'assertion de conservation des tokens dans `transfer()`
2. Ecrire l'assertion de non-négativité dans `withdraw()`

In [9]:
# Exercice 2 : SMTChecker - Assertions pour un TokenVault
# TODO etudiant : identifiez les assertions SMTChecker manquantes
TOKEN_VAULT_CODE = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract TokenVault {
    mapping(address => uint256) public balances;
    uint256 public totalSupply;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalSupply += msg.value;
    }

    function transfer(address to, uint256 amount) external {
        require(balances[msg.sender] >= amount);
        balances[msg.sender] -= amount;
        balances[to] += amount;
        // TODO etudiant : assert conservation des tokens (totalSupply inchange)
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount);
        balances[msg.sender] -= amount;
        totalSupply -= amount;
        // TODO etudiant : assert balance non negative
        payable(msg.sender).transfer(amount);
    }
}
"""

# TODO etudiant : listez les 2 assertions manquantes sous forme de commentaires Solidity
# Indice 1 : dans transfer(), totalSupply doit rester identique apres l'operation
# Indice 2 : dans withdraw(), la balance de msg.sender ne peut pas etre negative
# Etape 1 : ecrire assert(totalSupply == totalSupply) avant et apres transfer
# Etape 2 : ecrire assert(balances[msg.sender] >= 0) apres le decrement dans withdraw
missing_asserts = []  # TODO etudiant : ajoutez vos assertions en strings

print("Exercice a completer")
print("Identifiez les assertions SMTChecker manquantes dans le contrat TokenVault")

Exercice a completer
Identifiez les assertions SMTChecker manquantes dans le contrat TokenVault


***

## 4. Patterns Courants

### 4.1 Conservation des tokens (ERC-20)

In [10]:
# Specification ERC-20 simplifiee
ERC20_SPEC = '''
methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;
    require balanceOf(from) >= amount;
    
    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();
    
    env.msgSender = from;
    transfer(to, amount);
    
    assert balanceOf(from) == fromBalanceBefore - amount;
    assert balanceOf(to) == toBalanceBefore + amount;
    assert totalSupply() == supplyBefore;
}
'''

print("Specification ERC-20:")
print(ERC20_SPEC)

Specification ERC-20:

methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;
    require balanceOf(from) >= amount;

    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();

  

### Exercice 3 : CVL - Règle pour approve() et transferFrom()

L'exemple ERC-20 ci-dessus spécifie la règle `transferConservesTokens`.
Écrivez une règle CVL supplémentaire pour vérifier que `transferFrom` respecte les allowances.

**Objectif** : Completer la règle `transferFromRespectsAllowance` qui vérifie que
`transferFrom` réduit correctement l'allowance et déplace les tokens.

**Indice** :
- La fonction `transferFrom(owner, to, amount)` nécessite `allowance(owner, spender) >= amount`
- Vérifiez que l'allowance diminue du bon montant
- Vérifiez que les balances sont mises à jour correctement

**Étapes** :
1. Définir les preconditions (`require`) dans la règle CVL
2. Capturer les valeurs avant l'appel (balances, allowance)
3. Appeler `transferFrom` avec le bon `env.msgSender`
4. Ecrire les 3 assertions de vérification

In [11]:
# Exercice 3 : CVL - Regle pour approve() et transferFrom()
# TODO etudiant : completez la regle CVL transferFromRespectsAllowance
TRANSFERFROM_RULE = """
// TODO etudiant : completez la regle CVL ci-dessous

rule transferFromRespectsAllowance(address owner, address spender, address to, uint256 amount) {
    // Etape 1 : Preconditions
    // TODO etudiant : ajouter les require necessaires
    // require owner != to;
    // require balanceOf(owner) >= amount;
    // require allowance(owner, spender) >= amount;

    // Etape 2 : Capturer les valeurs avant l'appel
    // uint256 ownerBalBefore = balanceOf(owner);
    // uint256 toBalBefore = balanceOf(to);
    // uint256 allowanceBefore = allowance(owner, spender);

    // Etape 3 : Appeler transferFrom avec spender comme msgSender
    // env.msgSender = spender;
    // transferFrom(owner, to, amount);

    // Etape 4 : Verifier les resultats
    // TODO etudiant : ecrire les 3 assertions
    // assert balanceOf(owner) == ownerBalBefore - amount;
    // assert balanceOf(to) == toBalBefore + amount;
    // assert allowance(owner, spender) == allowanceBefore - amount;
}
"""

print("Exercice a completer")
print("Completez la regle CVL transferFromRespectsAllowance dans TRANSFERFROM_RULE")

Exercice a completer
Completez la regle CVL transferFromRespectsAllowance dans TRANSFERFROM_RULE


***

## 5. Exercices

In [12]:
# Exercice: Verifier un SimpleAuction
# TODO etudiant : ecrire les specifications CVL pour le contrat SimpleAuction
# Indice : quelles proprietes un enchere doit-elle garantir ?
# Etape 1 : definir les invariants (highestBid toujours positif, ended ne peut que passer a true)
# Etape 2 : ecrire une regle pour verifier que bid() met a jour highestBidder et highestBid

EXERCISE_AUCTION = '''
// Contrat
contract SimpleAuction {
    address public beneficiary;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    function bid() external payable {
        require(!ended, "Auction ended");
        require(msg.value > highestBid, "Bid too low");
        highestBidder = msg.sender;
        highestBid = msg.value;
    }

    function end() external {
        require(!ended, "Already ended");
        ended = true;
    }
}

// Specification CVL
// TODO: etudiant - definissez les invariants ci-dessous
/*
methods {
    // TODO: etudiant - declarer les fonctions du contrat
}

// Invariant: le highest bid est toujours positif ou nul
invariant highestBidPositive()
    // TODO: etudiant - ecrire la condition

// Invariant: ended ne peut que passer de false a true
invariant endedIsMonotonic()
    // TODO: etudiant - ecrire la condition
    // Indice : ended => toujours(ended)

// Regle: un bid valide met a jour le highest bidder
rule bidUpdatesHighestBidder(address bidder, uint256 amount) {
    // TODO: etudiant - ecrire la precondition (pas ended, amount > highestBid)
    // TODO: etudiant - appeler bid() avec les bons parametres env
    // TODO: etudiant - verifier highestBidder == bidder et highestBid == amount
}
*/
'''

print("Exercice Auction Formal Verification - a completer")
print("Implementez les specifications CVL marquees TODO dans EXERCISE_AUCTION")


Exercice Auction Formal Verification - a completer
Implementez les specifications CVL marquees TODO dans EXERCISE_AUCTION


**Indice** : Pour les invariants, réfléchissez aux propriétés qu'une enchère valide doit toujours respecter : le plus haut enchère est toujours positif, une enchère terminée ne peut pas reprendre. Pour la règle, vérifiez que `bid()` met correctement à jour `highestBidder` et `highestBid`.

***
## 6. Résumé

| Concept | Description |
|--------|-------------|
| Invariant | Propriété toujours vraie |
| Règle | Comportement attendu après une action |
| `methods` | Déclaration des fonctions du contrat (CVL) |
| `envfree` | Fonction sans effets de bord |
| `env` | Environnement (msg.sender, msg.value, block.timestamp) |
| `assert` | Assertion SMTChecker vérifiée exhaustivement |
| `pragma experimental SMTChecker` | Activation du model checker natif |

### Outils de vérification formelle

| Outil | Type | Coût | Complexité |
|-------|------|------|------------|
| **Certora Prover** | Externe (CVL) | Commercial | Élevée |
| **SMTChecker** | Natif Solidity | Gratuit | Moyenne |
| **Halmos** | Symbolic EVM | Open source | Élevée |

### Avantages
- Couverture complète (tous les cas possibles)
- Preuves mathématiques rigoureuses
- Détection de bugs subtils (reentrancy, overflow)

### Limites
- Complexité de configuration (Certora)
- Temps de vérification
- Faux positifs possibles
- SMTChecker : pas de support complet des boucles

***

**Notebook suivant** : [Zero-Knowledge Proofs](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs-Python.ipynb)

## Résumé et perspectives

Ce notebook a introduit la vérification formelle des smart contracts, une approche qui complète les tests unitaires et le fuzz testing en apportant des preuves mathématiques exhaustives. Nous avons exploré le langage CVL (Certora Vérification Language) pour ecrire des invariants (conservation des tokens, non-négativité des soldes) et des règles de comportement sur un contrat Vault, puis découvert SMTChecker, l'alternative native et gratuite intégrée au compilateur Solidity. La demonstration du contrat VaultBuggy a illustré la puissance de l'approche : SMTChecker produit un contre-exemple concret quand une assertion peut être violée. La démonstration ci-dessus a été exécutée avec solc 0.8.28 sous Linux et z3 4.12 : les trois conditions de VaultSMT sont prouvées, et les deux contrats fautifs produisent des contre-exemples réels.

La vérification formelle reste un investissement significatif en temps et en expertise, mais elle est indispensable pour les contrats critiques qui sécurisent des milliards de dollars en valeur verrouillée (DeFi, bridges, protocoles de gouvernance). Les outils évoluent rapidement : Certora Prover pour les spécifications expressives en CVL, Halmos pour l'exécution symbolique sur l'EVM, et SMTChecker pour les assertions natives sans dépendance externe. La complémentarité avec les tests classiques (foundry test, fuzz testing, invariant testing) définit une stratégie de validation multicouche ou chaque méthode couvre les angles morts des autres.

Le prochain notebook aborde les preuves à divulgation nulle (Zero-Knowledge Proofs), une technique cryptographique fondamentale pour la confidentialité sur blockchain : prouver qu'un énoncé est vrai sans révéler aucune information supplémentaire. Ce concept, au coeur des zk-SNARKs et des zk-rollups, constitue l'un des domaines les plus actifs de la recherche blockchain actuelle : [SC-15-Zero-Knowledge-Proofs-Python](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs-Python.ipynb).

***

[<< Précédent : Fuzz & Invariants](SC-13-Fuzz-Invariants-Python.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Zero-Knowledge Proofs >>](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs-Python.ipynb)